## Introduction and Definitions

This will be an exploration data analysis of the different physical characteristics of stars in the near Milky Way galaxy.

We will make use of the Hipparcos catalog as our main dataset. Hipparcos is a european astronomy satellite that was operational between from 1989 to 1993 and recorded observations of ~118k stellar objects. The dataset is publicly available from multiple sources, including the Stasbourg Astronomical Data Center (https://cds.unistra.fr/).

Unfortunately there's a catch. Space is huge, and it's hard to see faraway objects in the dark. As a result the Hipparcos catalog is heavily biased towards more luminous stars, since they're easier to see. So we're not going to get a very good representative sample of lower-luminosity objects unless we bring in another dataset.

Luckily we have very descriptively named "Woolley Catalog of Stars within 25 Parsecs" to supplement. This won't feature many of the bright stars found in hipparcos, but it will have a much higher proportion of dim stars and should give us enough of a sample size to work with.

*I had hoped to get the Gleise dataset to work instead since it's much more recent, but Woolley was the first near-star survey I found and turns out it's the most complete.*

We will quantify the approximate ranges of several characteristics across different stellar classifications:

**Direct Measurements**
- Color (B-V)
- Apparent Magnitude

**Calculated Measures**
- Temperature (Derived from Color)
- Luminosity or Absolute Magnitude (Derived from Apparent Magnitude and Distance)

Analysis will be done across two categories of classification, **Spectral Class** and **Luminosity Class**

Spectral Class is denoted by a letter that represents the approximate temperature of the star, from hottest to coolest they are (O, B, A, F, G, K, and M). The wavelength of thermal radiation (light) is inversly proportional to the temperature emmiting it, so hot stars will appear more blue and cooler stars more red. These categories are then subdivided 0-9 for the more specific temperature classification (0 is hottest, 9 is coolest) 

Luminosity Class is determined by it's luminosity (quantity of emitted radiation) as well as it's temperature. Note that temperature is relevant to both classifications - this will be important later. The classes are:
- Ia:   Hypergiants
- Ib:   Supergiants
- II:   Bright Giants
- III:  Giants
- IV:   Subgiants
- V:    Main Sequence / Red Dwarfs
- VI:   Subdwarfs
- VII:  White Dwarfs

The Sun, for example, is a relatively hot main sequence star and is classified as a type **G2V**. Polaris (the North Star) is actually a trinary system composed of the F7Ib Supergiant Polaris A and it's two smaller companions (F3V and F6V)

Ok enough trivia. Now that you understand the basics lets get down to business...

In [9]:
import urllib.request
import csv
import sqlite3
import io


def download_and_load_data(url, db, target_table):
    print("Downloading...")
    with urllib.request.urlopen(url) as resp:
        raw = resp.read().decode("utf-8")

    
    lines = [l for l in raw.splitlines() if not l.startswith("#")]
    reader = csv.DictReader(io.StringIO("\n".join(lines)))

    cols = reader.fieldnames
    expected = len(cols)
    print(f"Columns ({expected}): {cols}")

    rows = []
    for i, row in enumerate(reader):
        # DictReader puts overflow columns under None key — drop them
        row.pop(None, None)
        if len(row) != expected:
            print(f"Skipping row {i}: got {len(row)} cols")
            continue
        rows.append([row[c] for c in cols])

    print(f"Parsed {len(rows)} valid rows")

    col_defs = ", ".join(f'"{c.strip().replace('-', '_').lower()}" TEXT' for c in cols)
    placeholders = ", ".join("?" * expected)

    con = sqlite3.connect(db)

    # Try except so if an error occurs we don't have to restart kernal because of a connection left open
    try:
        cur = con.cursor()
        cur.execute(f'DROP TABLE IF EXISTS "{target_table}"')
        cur.execute(f'CREATE TABLE "{target_table}" ({col_defs})')
        cur.executemany(f'INSERT INTO "{target_table}" VALUES ({placeholders})', rows)
        con.commit()
        con.close()
        print(f"Loaded {len(rows)} rows into {db} table '{target_table}'")
    except Exception as e:
        print(e)
        con.close()


In [10]:
URL_HIPPARCOS = (
    "https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync"
    "?REQUEST=doQuery&LANG=ADQL&FORMAT=csv"
    '&QUERY=SELECT+*+FROM+"I/239/hip_main"'
)

DB= 'stellar_analysis.db'
TARGET_TABLE_HIP = 'hipparcos_bronze'

download_and_load_data(URL_HIPPARCOS, DB, TARGET_TABLE_HIP)

Downloading...
Columns (79): ['recno', 'HIP', 'Proxy', 'RAhms', 'DEdms', 'Vmag', 'VarFlag', 'r_Vmag', 'RAICRS', 'DEICRS', 'AstroRef', 'Plx', 'pmRA', 'pmDE', 'e_RAICRS', 'e_DEICRS', 'e_Plx', 'e_pmRA', 'e_pmDE', 'DE:RA', 'Plx:RA', 'Plx:DE', 'pmRA:RA', 'pmRA:DE', 'pmRA:Plx', 'pmDE:RA', 'pmDE:DE', 'pmDE:Plx', 'pmDE:pmRA', 'F1', 'F2', 'BTmag', 'e_BTmag', 'VTmag', 'e_VTmag', 'm_BTmag', 'B-V', 'e_B-V', 'r_B-V', 'V-I', 'e_V-I', 'r_V-I', 'CombMag', 'Hpmag', 'e_Hpmag', 'Hpscat', 'o_Hpmag', 'm_Hpmag', 'Hpmax', 'HPmin', 'Period', 'HvarType', 'moreVar', 'morePhoto', 'CCDM', 'n_CCDM', 'Nsys', 'Ncomp', 'MultFlag', 'Source', 'Qual', 'm_HIP', 'theta', 'rho', 'e_rho', 'dHp', 'e_dHp', 'Survey', 'Chart', 'Notes', 'HD', 'BD', 'CoD', 'CPD', '(V-I)red', 'SpType', 'r_SpType', '_RA_icrs', '_DE_icrs']
Parsed 118218 valid rows
Loaded 118218 rows into stellar_analysis.db table 'hipparcos_bronze'


In [12]:
import pandas as pd
# Helper function
def query_sqlite(query, db):
    con = sqlite3.connect(db)
    df = pd.read_sql(query, con)
    con.close()
    return df

# Quick sanity check to make sure everything worked correctly
df = query_sqlite('SELECT * FROM hipparcos_bronze', DB)

# Lots of different stellar designations to parse out
print(df['sptype'].unique().tolist()[:30])

['F5', 'K3V', 'B9', 'F0V', 'G8III', 'M0V:', 'G0', 'M6e-M8.5e Tc', 'G5', 'F6V', 'A2', 'K4III', 'K0III', 'K0', 'K2', 'F3V', ' ', 'K5', 'G8/K0III/IV', 'F2V', 'G0V', 'G3IV', 'F7V', 'G5V', 'F3/F5V', 'A0', 'B8', 'F2', 'F7.5IV-V', 'G6V']


In [13]:
from stellar_parser import parse_stellar_column

# TY Claude I owe you my life
def build_silver_table(db_path, source_table_name, table_name, source_cols):
    con = sqlite3.connect(db_path)

    # Load only the columns we need from source
    src_cols_sql = ", ".join(f'"{c}"' for c in source_cols)
    df = pd.read_sql(f"SELECT {src_cols_sql} FROM {source_table_name}", con)

    # Convert column names to lowercase and SQL-safe
    df.columns = [c.lower().replace("-", "_") for c in df.columns]
    col_types = {k.lower().replace("-", "_"): v for k, v in source_cols.items()}

    # Cast columns to their specified types
    for col, dtype in col_types.items():
        if dtype == "REAL":
            # Strip whitespace and non-numeric characters (preserve decimal points)
            df[col] = df[col].astype(str).str.strip().str.replace(r"[^\d.]", "", regex=True)
            df[col] = pd.to_numeric(df[col], errors="coerce")

        elif dtype == "INTEGER":
            df[col] = df[col].astype(str).str.strip().str.replace(r"[^\d]", "", regex=True)
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        # TEXT columns need no conversion

    # Parse Sp -> sp_class, lum_class
    parsed = parse_stellar_column(df["sptype"])
    df["sp_class"] = parsed["spectral_type"]
    df["lum_class"] = parsed["lum_class"]

    # Sort by plx descending (closest stars first); nulls last
    df.sort_values("plx", ascending=False, na_position="last", inplace=True)

    # Write silver table
    df.to_sql(table_name, con, if_exists="replace", index=False)

    cur = con.cursor()
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_sp_class ON {table_name}(sp_class)')
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_sp_plx ON {table_name}(sp_class, plx DESC)')
    cur.execute(f"ALTER TABLE {table_name} ADD COLUMN src TEXT DEFAULT '{source_table_name.split('_')[0]}'")
    con.commit()

    row_count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table_name}", con).iloc[0]["n"]
    print(f"Created '{table_name}' with {row_count:,} rows.")
    con.close()


source_cols_hip = {
    "ccdm":     "TEXT",
    "plx":      "REAL",
    "e_plx":    "REAL",
    "vmag":     "REAL",
    "b_v":      "REAL",
    "e_b_v":    "REAL",
    "sptype":   "TEXT",
}

build_silver_table(DB, 'hipparcos_bronze', 'hipparcos_silver', source_cols_hip)


Created 'hipparcos_silver' with 118,218 rows.


In [14]:
df = query_sqlite('SELECT * FROM hipparcos_silver LIMIT 5000', DB)

# Lots of different stellar designations to parse out
df.head(20)

,ccdm,plx,e_plx,vmag,b_v,e_b_v,sptype,sp_class,lum_class,src
0,14396-6050,772.33,2.42,11.01,1.807,0.020,M5Ve,M,V,hipparcos
1,14396-6050,742.12,1.40,1.35,0.900,0.020,K1V,K,V,hipparcos
2,14396-6050,742.12,1.40,0.01,0.710,0.040,G2V,G,V,hipparcos
3,17578+0441,549.01,1.58,9.54,1.570,0.015,sdM4,M,NaN,hipparcos
4,11033+3558,392.40,0.91,7.49,1.502,0.014,M2V,M,V,hipparcos
5,06451-1643,379.21,1.58,1.44,0.009,0.007,A0m...,A,NaN,hipparcos
6,,336.48,1.82,10.37,1.510,0.510,M3.5Ve,M,V,hipparcos
7,03329-0927,310.75,0.85,3.72,0.881,0.007,K2V,K,V,hipparcos
8,,303.90,0.87,7.35,1.483,0.003,M2/M3V,M,V,hipparcos
9,,299.58,2.20,11.12,1.746,0.020,M4.5V,M,V,hipparcos


In [15]:
query_class_counts = """
SELECT
    lum_class,
    COUNT(CASE WHEN sp_class = 'O' THEN 1 END) AS O,
    COUNT(CASE WHEN sp_class = 'B' THEN 1 END) AS B,
    COUNT(CASE WHEN sp_class = 'A' THEN 1 END) AS A,
    COUNT(CASE WHEN sp_class = 'F' THEN 1 END) AS F,
    COUNT(CASE WHEN sp_class = 'G' THEN 1 END) AS G,
    COUNT(CASE WHEN sp_class = 'K' THEN 1 END) AS K,
    COUNT(CASE WHEN sp_class = 'M' THEN 1 END) AS M
FROM hipparcos_silver
GROUP BY lum_class
ORDER BY
    CASE lum_class
        WHEN 'I'   THEN 1
        WHEN 'II'  THEN 2
        WHEN 'III' THEN 3
        WHEN 'IV'  THEN 4
        WHEN 'V'   THEN 5
        WHEN 'VI'  THEN 6
        ELSE 7
END
"""

df = query_sqlite(query_class_counts, DB)

# Lots of different stellar designations to parse out
df.head(20)

,lum_class,O,B,A,F,G,K,M
0,I,42,460,105,170,171,92,69
1,II,11,526,147,195,333,459,80
2,III,18,1407,786,640,4056,12907,1943
3,IV,8,1170,1347,1941,1398,524,5
4,V,62,3496,4444,9219,5593,1643,216
5,NaN,129,3364,11877,13480,11238,16413,2479
6,VII,0,0,0,0,1,0,0


## Placeholder - Talk about Luminosity bias

Why we need a second dataset

In [16]:
# Woolley Catalog of Stars within 25 Parsecs
URL_WOOLLEY = (
    "https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync"
    "?REQUEST=doQuery&LANG=ADQL&FORMAT=csv"
    '&QUERY=SELECT+*+FROM+"V/32A/catalog"'
)
TARGET_TABLE_WLY = "woolley_bronze"

download_and_load_data(URL_WOOLLEY, DB, TARGET_TABLE_WLY)

Downloading...
Columns (37): ['recno', 'Woolley', 'm_Woolley', 'plx', 'e_plx', 'n_plx', 'pmRA', 'pmDE', 'RVel', 'n_RVel', 'U', 'V', 'W', 'GCdist', 'e', 'i', 'LC_Code', 'SpType', 'r_SpType', 'Mag', 'n_Mag', 'B-V', 'U-B', 'Mv', 'RAB1950', 'DEB1950', 'GCTP', 'HD', 'DM', 'GCRV', 'PM_Name', 'HR', 'Vys', 'Remark1', 'Remark2', '_RA_icrs', '_DE_icrs']
Parsed 2150 valid rows
Loaded 2150 rows into stellar_analysis.db table 'woolley_bronze'


In [17]:
con = sqlite3.connect(DB)
df = pd.read_sql('SELECT b_v, plx, sptype, lc_code, * FROM woolley_bronze WHERE b_v >0 LIMIT 5000', con)
con.close()

# Thank god. We have parallax, bv color, and classification. Luminosity is it's own thing as LC code which I think is just
# an integer for the roman numeral designation. We can work with this
df.head(20)

,b_v,plx,sptype,lc_code,recno,woolley,m_woolley,plx,e_plx,n_plx,...,hd,dm,gcrv,pm_name,hr,vys,remark1,remark2,_ra_icrs,_de_icrs
0,1.36,63,M0,5,272,123,,63,5,9,...,19305,BD+01 543,1721,CC 207,,98,,,46.61075111111111,1.9648544444444442
1,0.68,49,G0,5,295,9114,A,49,10,,...,20727,BD+08 496,1850,3998,,,,,50.15323166666666,9.03366611111111
2,0.86,42,K1,5,284,9112,,42,4,9,...,20165,BD+08 482,1789,3874,,,,,48.69554833333333,8.981001666666666
3,1.61,144,M4,,223,105,B,144,4,9,...,,,,L10859,,396,,,39.065688888888886,6.872398333333332
4,0.97,144,K3,5,222,105,A,144,4,9,...,16160,BD+06 398,1464,3121,753,,4,,39.01994388888888,6.887523333333332
5,0.31,43,F0,4,245,9099,,43,10,,...,17094,BD+09 359,1553,3309,813,,2,,41.23515138888888,10.114969166666665
6,1.27,59,K6,5,252,114,,59,21,,...,17660,BD+15 395,1592,3407,,,,,42.652067222222215,15.710326388888888
7,1.18,45,K8,5,328,9128,,45,S,,...,,BD+11 514,,L11244,,429,,,56.213939999999994,11.919640555555555
8,0.3,72,F,7,327,151,,72,11,,...,,,,CC 257,,,,,56.14972194444444,18.435780277777774
9,0.62,69,G1,5,356,160,,69,5,9,...,25680,BD+21 587,2331,4913,1262,,,,61.33290249999999,22.008209166666663


In [18]:
# I can't believe that worked the first try.
source_cols_wly = {
    "recno":    "TEXT",
    "plx":      "REAL",
    "e_plx":    "REAL",
    "B_V":      "REAL",
    "Mv":       "REAL",
    "SpType":   "TEXT",
    "LC_Code":  "TEXT"
}

build_silver_table(DB, 'woolley_bronze', 'woolley_silver', source_cols_wly)

lum_update_sql = '''
    UPDATE woolley_silver
    SET lum_class = CASE
        WHEN lc_code = 1 THEN 'I'
        WHEN lc_code = 2 THEN 'II'
        WHEN lc_code = 3 THEN 'III'
        WHEN lc_code = 4 THEN 'IV'
        WHEN lc_code = 5 THEN 'V'
        WHEN lc_code = 6 THEN 'VI'
        WHEN lc_code = 7 THEN 'VII'
        ELSE '' END;
'''

# Handle lc_code convert it to lum_class
con = sqlite3.connect(DB)

try:
    cur = con.cursor()

    cur.execute(lum_update_sql)
    cur.execute('ALTER TABLE woolley_silver DROP COLUMN lc_code;')

    con.commit()
    con.close()
except Exception as e:
    print(e)
    con.close()

Created 'woolley_silver' with 2,150 rows.


In [23]:
df = query_sqlite('SELECT * FROM woolley_silver LIMIT 5000', DB)

In [24]:
df.head()

,recno,plx,e_plx,b_v,mv,sptype,sp_class,lum_class,src
0,1256,761.0,5.0,NaN,15.11,M5E,M,V,woolley
1,1270,743.0,7.0,0.88,5.68,K0,K,V,woolley
2,1269,743.0,7.0,0.68,4.34,G2,G,V,woolley
3,1597,548.0,3.0,1.74,13.23,M5,M,V,woolley
4,904,429.0,8.0,2.01,16.69,M6E,M,V,woolley


In [25]:
df = query_sqlite('SELECT * FROM hipparcos_silver LIMIT 5000', DB)
df.head()

,ccdm,plx,e_plx,vmag,b_v,e_b_v,sptype,sp_class,lum_class,src
0,14396-6050,772.33,2.42,11.01,1.807,0.020,M5Ve,M,V,hipparcos
1,14396-6050,742.12,1.40,1.35,0.900,0.020,K1V,K,V,hipparcos
2,14396-6050,742.12,1.40,0.01,0.710,0.040,G2V,G,V,hipparcos
3,17578+0441,549.01,1.58,9.54,1.570,0.015,sdM4,M,NaN,hipparcos
4,11033+3558,392.40,0.91,7.49,1.502,0.014,M2V,M,V,hipparcos


## Building a gold table

In [22]:
sql_gold = '''
    CREATE TABLE stars AS
    WITH hip_tmp AS (
        SELECT
            1000/(plx * 1.0) dist_pc                        -- Distance in parsecs (derived from trigonometric parallax)
            ,vmag - 5 * LOG10(1000/(plx * 1.0)) + 5 abs_mag -- Absolute magnitude
            ,b_v                                            -- BV Color
            ,sp_class                                       -- Spectral Class
            ,lum_class                                      -- Luminosity Class
            ,src                                            -- Source Dataset
        FROM hipparcos_silver
        WHERE 1=1
            -- parralax quality assurance. Maximum standard error of 10% is arbitrary
            AND plx IS NOT NULL
            AND plx > 0

            -- visual magnitude quality assurance 
            -- No error measure here so fingers crossed I guess
            AND vmag IS NOT NULL

            -- bv color quality assurance
            AND b_v IS NOT NULL
            -- AND e_b_v < 0.1

            -- Only stars where we have info on both spectral and luminosity class
            AND sp_class IS NOT NULL
            AND lum_class IS NOT NULL
    )
    ,wly_tmp AS (
        SELECT
            1000/(plx * 1.0) dist_pc                        -- Distance in parsecs (derived from trigonometric parallax)
            ,mv abs_mag                                     -- Absolute magnitude
            ,b_v                                            -- BV Color
            ,sp_class                                       -- Spectral Class
            ,lum_class                                      -- Luminosity Class
            ,src                                            -- Source Dataset
        FROM woolley_silver
        WHERE 1=1
            -- parralax quality assurance. Maximum standard error of 10 percent is arbitrary
            AND plx IS NOT NULL
            AND plx > 0

            -- visual magnitude quality assurance 
            -- No error measure here so fingers crossed I guess
            AND abs_mag IS NOT NULL

            -- bv color quality assurance
            AND b_v IS NOT NULL

            -- Only stars where we have info on both spectral and luminosity class
            AND sp_class IS NOT NULL
            AND lum_class IS NOT NULL
            AND lum_class <> ''
    )
    ,combined AS (
        SELECT * FROM hip_tmp
        UNION ALL
        SELECT * FROM wly_tmp
    )
    SELECT * FROM combined ORDER BY dist_pc ASC
'''

con = sqlite3.connect(DB)
try:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS stars")

    # Build final clean table
    cur.execute(sql_gold)

    # Index on classes
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_stars_sp_class ON stars (sp_class)')
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_stars_lum_class ON stars (lum_class)')

    # Composite index on class, order by distance ascending
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_stars_sp_dist ON stars (sp_class, dist_pc ASC)')
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_stars_lum_dist ON stars (lum_class, dist_pc ASC)')

    con.commit()
    con.close()
except Exception as e:
    print(e)
    con.close()


In [ ]:
df = query_sqlite('SELECT * FROM stars LIMIT 1000', DB)
df.head(20)

,dist_pc,abs_mag,b_v,sp_class,lum_class,src
0,1.294783,15.449015,1.807,M,V,hipparcos
1,1.345895,5.680000,0.880,K,V,woolley
2,1.345895,4.340000,0.680,G,V,woolley
3,1.347491,5.702371,0.900,K,V,hipparcos
4,1.347491,4.362371,0.710,G,V,hipparcos
5,1.824818,13.230000,1.740,M,V,woolley
6,2.331002,16.690000,2.010,M,V,woolley
7,2.525253,10.480000,1.510,M,V,woolley
8,2.548420,10.458645,1.502,M,V,hipparcos
9,2.652520,11.180000,0.120,A,VII,woolley


## 1: Demonstrating Luminosity Bias

We're going to look at two different cross-sections to demonstrate the impact of luminosity bias in our dataset.

The first is a breakdown of the the different class combinations as a % of total records by source (Woolley or Hipparcos). Since the woolley dataset is restricted to only nearby stars, we expect that it will show a much larger proportion of lower-luminosity stars. Hipparcos on the other hand should heavily over-represent large and bright object.

Second, we'll be looking at groupings by distance. This will give us a more granular view of what's going on.

Excluding subdwarfs and white dwarfs for the moment for simplicity's sake.

In [26]:
# Get counts by source and class. Then create a table that shows % of total by this breakdown.
query_by_src = """
WITH totals AS (
    SELECT src, COUNT(*) AS total
    FROM stars
    GROUP BY src
),
counts AS (
    SELECT
        src,
        lum_class,
        COUNT(CASE WHEN sp_class = 'O' THEN 1 END) AS O,
        COUNT(CASE WHEN sp_class = 'B' THEN 1 END) AS B,
        COUNT(CASE WHEN sp_class = 'A' THEN 1 END) AS A,
        COUNT(CASE WHEN sp_class = 'F' THEN 1 END) AS F,
        COUNT(CASE WHEN sp_class = 'G' THEN 1 END) AS G,
        COUNT(CASE WHEN sp_class = 'K' THEN 1 END) AS K,
        COUNT(CASE WHEN sp_class = 'M' THEN 1 END) AS M
    FROM stars
    GROUP BY src, lum_class
)
SELECT
    c.src,
    c.lum_class,
    ROUND(100.0 * c.O / t.total, 2) AS O,
    ROUND(100.0 * c.B / t.total, 2) AS B,
    ROUND(100.0 * c.A / t.total, 2) AS A,
    ROUND(100.0 * c.F / t.total, 2) AS F,
    ROUND(100.0 * c.G / t.total, 2) AS G,
    ROUND(100.0 * c.K / t.total, 2) AS K,
    ROUND(100.0 * c.M / t.total, 2) AS M
FROM counts c
JOIN totals t ON c.src = t.src
WHERE lum_class NOT IN ('D', 'VII', 'VI')
ORDER BY
    c.src,
    CASE c.lum_class
        WHEN 'I'   THEN 1
        WHEN 'II'  THEN 2
        WHEN 'III' THEN 3
        WHEN 'IV'  THEN 4
        WHEN 'V'   THEN 5
        WHEN 'VI'  THEN 6
        ELSE 7
    END
"""

query_by_dist = """
WITH totals AS (
    SELECT
        (CAST(dist_pc / 50 AS INT) * 50) AS dist_bucket,
        COUNT(*) AS total
    FROM stars
    WHERE dist_pc IS NOT NULL
    GROUP BY dist_bucket
),
counts AS (
    SELECT
        (CAST(dist_pc / 50 AS INT) * 50) AS dist_bucket,
        COUNT(CASE WHEN lum_class = 'I'   THEN 1 END) AS I,
        COUNT(CASE WHEN lum_class = 'II'  THEN 1 END) AS II,
        COUNT(CASE WHEN lum_class = 'III' THEN 1 END) AS III,
        COUNT(CASE WHEN lum_class = 'IV'  THEN 1 END) AS IV,
        COUNT(CASE WHEN lum_class = 'V'   THEN 1 END) AS V
    FROM stars
    WHERE dist_pc IS NOT NULL
    GROUP BY dist_bucket
)
SELECT
    c.dist_bucket,
    ROUND(100.0 * c.I   / t.total, 2) AS I,
    ROUND(100.0 * c.II  / t.total, 2) AS II,
    ROUND(100.0 * c.III / t.total, 2) AS III,
    ROUND(100.0 * c.IV  / t.total, 2) AS IV,
    ROUND(100.0 * c.V   / t.total, 2) AS V,
    t.total
FROM counts c
JOIN totals t ON c.dist_bucket = t.dist_bucket
ORDER BY c.dist_bucket
"""


In [27]:
df_src = query_sqlite(query_by_src, DB)
df_src.head(20)

,src,lum_class,O,B,A,F,G,K,M
0,hipparcos,I,0.08,0.82,0.19,0.30,0.31,0.17,0.12
1,hipparcos,II,0.02,0.94,0.26,0.35,0.60,0.82,0.14
2,hipparcos,III,0.03,2.53,1.41,1.15,7.28,23.19,3.50
3,hipparcos,IV,0.01,2.10,2.42,3.49,2.51,0.94,0.01
4,hipparcos,V,0.11,6.27,7.97,16.55,10.01,2.91,0.39
5,woolley,II,0.00,0.00,0.00,0.00,0.08,0.08,0.08
6,woolley,III,0.00,0.00,0.41,0.66,0.74,2.64,0.33
7,woolley,IV,0.00,0.00,0.33,3.22,3.05,0.99,0.00
8,woolley,V,0.00,0.33,3.54,11.54,20.12,26.79,21.19


In [28]:
df_src = query_sqlite(query_by_dist, DB)
df_src.head(20)

,dist_bucket,I,II,III,IV,V,total
0,0,0.04,0.25,6.73,10.15,81.87,4887
1,50,0.10,0.66,14.06,12.45,72.73,8206
2,100,0.08,1.53,23.94,13.22,61.22,8500
3,150,0.37,2.14,37.62,13.64,46.23,6818
4,200,0.59,3.14,49.36,12.19,34.71,5292
5,250,0.74,3.44,56.35,10.41,29.06,4332
6,300,1.13,3.79,59.13,10.28,25.67,3269
7,350,1.55,5.08,58.78,9.77,24.82,2518
8,400,2.39,5.16,60.13,9.73,22.59,1881
9,450,2.78,5.36,57.57,9.91,24.37,1473


As distance increase the percentage of main sequence (Class V) stars rapidly decreases, from about 80% within 50pc distance down to about 20% after 500 pc. This is why it's so important that we also include the woolley data in our analysis. Without it we simply can't get a representative sample of less luminous main sequence stars.

If we look far out into the night sky it's mostly the outliers that are bright enough for us to see.